May return to this notebook periodically for ongoing optimization.

In [17]:
import pandas as pd

Import Round one statistics - These are basic models with no GridSearch optimization and no feature selection. 

We will average them out, see which features are least correlated with errors, and use those figures to select a new feature set.

In [18]:
df_round1_linreg = pd.read_csv('models_daily_results/Round_1_Linreg.csv')
df_round1_rfr = pd.read_csv('models_daily_results/Round_1_RFR.csv')

In [19]:
dfs_to_merge = df_round1_linreg, df_round1_rfr

In [20]:
df_round1_rfr

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,close_lag1,close_lag2,return_lag1,next_close,next_close_change,next_close_change_pct,pred_next_close_change_pct,pred_error,abs_pred_error,directionally_correct
0,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,37.78,37.15,0.016958,35.71,-2.28,-6.001579,-6.03,-2.842064e-02,2.842064e-02,1
1,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,37.99,37.78,0.005558,36.49,0.78,2.184262,2.16,-2.426211e-02,2.426211e-02,1
2,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,35.71,37.99,-0.060016,36.24,-0.25,-0.685119,-0.69,-4.880789e-03,4.880789e-03,1
3,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,36.49,35.71,0.021843,36.00,-0.24,-0.662252,-0.66,2.251656e-03,2.251656e-03,1
4,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,36.24,36.49,-0.006851,35.91,-0.09,-0.250000,-0.25,9.436896e-15,9.436896e-15,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99245,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,123.54,123.78,-0.001939,126.23,0.74,0.589688,0.59,3.115786e-04,3.115786e-04,1
99246,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,125.49,123.54,0.015784,125.98,-0.25,-0.198051,-0.20,-1.948824e-03,1.948824e-03,1
99247,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,126.23,125.49,0.005897,126.41,0.43,0.341324,0.34,-1.324020e-03,1.324020e-03,1
99248,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,125.98,126.23,-0.001981,125.82,-0.59,-0.466735,-0.47,-3.264773e-03,3.264773e-03,1


In [21]:
df_round1_rfr.columns

Index(['timestamp', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'vwap',
       'transactions', 'close_vs_open_pct', 'daily_high_vs_low_pct',
       'prev_day_return', 'SMA10', 'SMA20', 'SMA50', 'SMA100', 'RSI', 'MACD',
       'MACD_signal', 'MACD_hist', 'rolling_volatility', 'close_lag1',
       'close_lag2', 'return_lag1', 'next_close', 'next_close_change',
       'next_close_change_pct', 'pred_next_close_change_pct', 'pred_error',
       'abs_pred_error', 'directionally_correct'],
      dtype='object')

In [27]:
from functools import reduce
import pandas as pd

# Load datasets
df_round1_linreg = pd.read_csv('models_daily_results/Round_1_Linreg.csv')
df_round1_rfr = pd.read_csv('models_daily_results/Round_1_RFR.csv')
df_round1_svr = pd.read_csv('models_daily_results/Round_1_SVR.csv')


# Create dictionary of dataframes to merge
dfs_to_merge = {
    "RFR": df_round1_rfr,
    "Linreg": df_round1_linreg,
    "SVR": df_round1_svr,
}

merge_cols = ["timestamp", "ticker"]

pred_cols = [
    "pred_next_close",
    "pred_next_close_change_pct",
    "pred_error",
    "abs_pred_error",
    "directionally_correct",
]

def prepare_model_df(df, model_name):

    df = df.copy()

    rename_map = {
        col: f"{model_name}_{col}"
        for col in pred_cols
        if col in df.columns
    }

    df = df.rename(columns=rename_map)

    model_cols = list(rename_map.values())

    return df[merge_cols + model_cols]

# Use first dataframe as base
first_key = list(dfs_to_merge.keys())[0]

base_df = dfs_to_merge[first_key].copy()

# Remove original un-prefixed prediction columns
base_df = base_df.drop(
    columns=[col for col in pred_cols if col in base_df.columns]
)

# Create model-specific dfs
model_dfs = [
    prepare_model_df(df, model_name)
    for model_name, df in dfs_to_merge.items()
]

# Merge everything
merged_results = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=merge_cols,
        how="inner"
    ),
    [base_df] + model_dfs
)

print("Merged shape:", merged_results.shape)

merged_results.head()

Merged shape: (99250, 39)


,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,RFR_abs_pred_error,RFR_directionally_correct,Linreg_pred_next_close_change_pct,Linreg_pred_error,Linreg_abs_pred_error,Linreg_directionally_correct,SVR_pred_next_close_change_pct,SVR_pred_error,SVR_abs_pred_error,SVR_directionally_correct
0,2025-01-02,AA,38.165,39.040,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,2.842064e-02,1,-1.20,4.801579,4.801579,1,0.02,6.021579,6.021579,0
1,2025-01-03,AA,37.950,37.950,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,2.426211e-02,1,0.38,-1.804262,1.804262,1,0.04,-2.144262,2.144262,1
2,2025-01-06,AA,36.000,37.075,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,4.880789e-03,1,-0.00,0.685119,0.685119,0,0.04,0.725119,0.725119,0
3,2025-01-07,AA,36.890,37.300,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,2.251656e-03,1,-0.11,0.552252,0.552252,1,0.03,0.692252,0.692252,0
4,2025-01-08,AA,35.780,36.030,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,9.436896e-15,1,0.00,0.250000,0.250000,0,0.03,0.280000,0.280000,0


In [28]:
# Find all matching model result columns
pred_error_cols = [
    col for col in merged_results.columns
    if col.endswith("_pred_error")
]

abs_pred_error_cols = [
    col for col in merged_results.columns
    if col.endswith("_abs_pred_error")
]

direction_cols = [
    col for col in merged_results.columns
    if col.endswith("_directionally_correct")
]

# Create aggregate dataframe
model_row_aggs = merged_results.copy()

model_row_aggs["avg_pred_error"] = model_row_aggs[pred_error_cols].mean(axis=1)

model_row_aggs["avg_abs_pred_error"] = model_row_aggs[abs_pred_error_cols].mean(axis=1)

model_row_aggs["pct_directionally_correct"] = (
    model_row_aggs[direction_cols].mean(axis=1) * 100
)

model_row_aggs

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,Linreg_pred_error,Linreg_abs_pred_error,Linreg_directionally_correct,SVR_pred_next_close_change_pct,SVR_pred_error,SVR_abs_pred_error,SVR_directionally_correct,avg_pred_error,avg_abs_pred_error,pct_directionally_correct
0,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,4.801579,4.801579,1,0.02,6.021579,6.021579,0,3.607720,3.617193,66.666667
1,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,-1.804262,1.804262,1,0.04,-2.144262,2.144262,1,0.000000,1.324262,100.000000
2,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,0.685119,0.685119,0,0.04,0.725119,0.725119,0,0.470079,0.471706,33.333333
3,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,0.552252,0.552252,1,0.03,0.692252,0.692252,0,0.415585,0.415585,66.666667
4,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,0.250000,0.250000,0,0.03,0.280000,0.280000,0,0.176667,0.176667,33.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99245,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,-0.179688,0.179688,1,0.02,-0.569688,0.569688,1,0.000104,0.249896,100.000000
99246,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,0.058051,0.058051,1,0.03,0.228051,0.228051,0,0.095367,0.096017,66.666667
99247,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,-0.131324,0.131324,1,0.03,-0.311324,0.311324,1,0.000000,0.147991,100.000000
99248,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,0.136735,0.136735,1,0.03,0.496735,0.496735,0,0.211157,0.212245,66.666667


# Feature quality analysis

Here we will find each feature's correlation to an absolute error. In this case, a smaller number is better.

Then we will find each feature's correlation to directional correctness. In this case, a larger number is better.

Then, I will calculate a metric I made up called feature_quaility, which is calculated as follows: 

    ((-abs_error_corr) * .7) + (direct_correct_corr  * .3)

I reversed the sign of abs_error_corr so that larger numbers would now be better, and weighed it 70%. direct_correct_corr was weighed 30%. The two numbers were added together to then obtain my final score.

In [29]:
# Build combined feature correlation dataframe
feature_corrs = pd.DataFrame({
    "feature": feature_cols,

    "corr_to_avg_abs_pred_error": [
        model_row_aggs[col].corr(
            model_row_aggs["avg_abs_pred_error"]
        )
        for col in feature_cols
    ],

    "corr_to_pct_directionally_correct": [
        model_row_aggs[col].corr(
            model_row_aggs["pct_directionally_correct"]
        )
        for col in feature_cols
    ]
})

# Reverse sign so:
# positive = GOOD
feature_corrs["good_abs_error_corr"] = (
    -feature_corrs["corr_to_avg_abs_pred_error"]
)

# Keep directional correctness positive
feature_corrs["good_direction_corr"] = (
    feature_corrs["corr_to_pct_directionally_correct"]
)

# Weighted composite score
feature_corrs["feature_quality_score"] = (
    (feature_corrs["good_abs_error_corr"] * 0.7) +
    (feature_corrs["good_direction_corr"] * 0.3)
)

# Sort best to worst
feature_corrs = feature_corrs.sort_values(
    "feature_quality_score",
    ascending=False
)

feature_corrs.head(30)

,feature,corr_to_avg_abs_pred_error,corr_to_pct_directionally_correct,good_abs_error_corr,good_direction_corr,feature_quality_score
23,next_close_change,-0.002978,0.335000,0.002978,0.335000,0.102584
22,next_close,-0.038965,0.064924,0.038965,0.064924,0.046753
2,low,-0.040434,0.055007,0.040434,0.055007,0.044806
12,SMA50,-0.038147,0.056690,0.038147,0.056690,0.043710
13,SMA100,-0.038147,0.056690,0.038147,0.056690,0.043710
5,vwap,-0.038842,0.054954,0.038842,0.054954,0.043676
3,close,-0.038904,0.054782,0.038904,0.054782,0.043667
19,close_lag1,-0.038662,0.055249,0.038662,0.055249,0.043638
0,open,-0.038616,0.055312,0.038616,0.055312,0.043625
20,close_lag2,-0.038627,0.054943,0.038627,0.054943,0.043522


Clear dropoff from volume onward.

In [14]:
row_agg_cols = [
    "timestamp",
    "ticker",
    "avg_pred_error",
    "avg_abs_pred_error",
    "pct_directionally_correct",
]

model_row_aggs_summary = model_row_aggs[row_agg_cols].copy()

model_row_aggs_summary.head()

,timestamp,ticker,avg_pred_error,avg_abs_pred_error,pct_directionally_correct
0,2025-01-02,AA,3.701579,3.701579,100.0
1,2025-01-03,AA,0.000000,1.094262,100.0
2,2025-01-06,AA,0.342560,0.715000,50.0
3,2025-01-07,AA,0.276126,0.605000,100.0
4,2025-01-08,AA,0.125000,0.580000,50.0
...,...,...,...,...,...
99245,2025-12-24,ZTS,0.030156,0.120000,100.0
99246,2025-12-26,ZTS,0.029026,0.280000,100.0
99247,2025-12-29,ZTS,0.079338,0.145000,100.0
99248,2025-12-30,ZTS,0.068368,0.195000,100.0
